In [1]:
# Notebook 全局导入与超参数。只改这一格，然后从上到下重新运行整个 notebook。
from pathlib import Path
import importlib.util
import sys
import warnings

from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold

from skopt import BayesSearchCV
from skopt.space import Categorical, Integer, Real
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

support_path_candidates = [
    Path("cuda_training_support.py"),
    Path("..") / "cuda_training_support.py",
]
SUPPORT_PATH = next((path.resolve() for path in support_path_candidates if path.exists()), None)
if SUPPORT_PATH is None:
    raise FileNotFoundError("找不到 cuda_training_support.py")

support_spec = importlib.util.spec_from_file_location("cuda_training_support", SUPPORT_PATH)
if support_spec is None or support_spec.loader is None:
    raise ImportError(f"无法加载训练辅助模块: {SUPPORT_PATH}")

cuda_training_support = importlib.util.module_from_spec(support_spec)
sys.modules["cuda_training_support"] = cuda_training_support
support_spec.loader.exec_module(cuda_training_support)

PROJECT_ROOT = SUPPORT_PATH.parent

# 如果内核从项目根目录启动，先移除本地 lightgbm 目录对官方包导入的遮蔽。
current_dir = Path.cwd().resolve()
if (current_dir / "lightgbm").is_dir():
    sys.path = [
        path
        for path in sys.path
        if Path(path or current_dir).resolve() != current_dir
    ]

import lightgbm as lgb

# NOTEBOOK_RANDOM_SEED: 控制数据切分、BayesSearch、Borderline-SMOTE、OOF、概率校准与模型训练的随机种子。
NOTEBOOK_RANDOM_SEED = 114514
# NOTEBOOK_TEST_SIZE: 固定留作最终评估的测试集比例。
NOTEBOOK_TEST_SIZE = 0.15
# NOTEBOOK_BAYES_N_ITER: BayesSearchCV 的候选参数组合采样次数。
NOTEBOOK_BAYES_N_ITER = 48
# NOTEBOOK_CV_FOLDS: 分层交叉验证折数。
NOTEBOOK_CV_FOLDS = 5
# NOTEBOOK_MODEL_N_JOBS: 单个 LightGBM 模型内部使用的线程数；默认给 4~16 个线程供 GPU 喂数。
NOTEBOOK_MODEL_N_JOBS = cuda_training_support.resolve_recommended_model_n_jobs()
# NOTEBOOK_SEARCH_N_JOBS: BayesSearchCV 并行 worker 数；同一张 GPU 上通常保持 1。
NOTEBOOK_SEARCH_N_JOBS = 1
# NOTEBOOK_SMOTE_K_NEIGHBORS: 每个训练折内部 Borderline-SMOTE 的少数类近邻数。
NOTEBOOK_SMOTE_K_NEIGHBORS = 3
# NOTEBOOK_SMOTE_SAMPLING_STRATEGY: Borderline-SMOTE 只把少数类扩到多数类的 75%，剩余差值交给 scale_pos_weight。
NOTEBOOK_SMOTE_SAMPLING_STRATEGY = 0.75
# NOTEBOOK_EARLY_STOPPING_ROUNDS: 每个训练折内部用于 early stopping 的 patience。
NOTEBOOK_EARLY_STOPPING_ROUNDS = 300
# NOTEBOOK_EARLY_STOPPING_VALIDATION_FRACTION: 每个训练折内部再切出的 early stopping 验证集比例。
NOTEBOOK_EARLY_STOPPING_VALIDATION_FRACTION = 0.15
# NOTEBOOK_BAYES_SCORING: BayesSearchCV 选参使用的目标指标。
NOTEBOOK_BAYES_SCORING = "roc_auc"
# NOTEBOOK_BAYES_VERBOSE: BayesSearchCV 日志级别。
NOTEBOOK_BAYES_VERBOSE = 0
# NOTEBOOK_TQDM_DESC: notebook 中训练进度条的标题。
NOTEBOOK_TQDM_DESC = "LightGBM 单路径 BayesSearchCV"
# NOTEBOOK_INITIAL_THRESHOLD: 阈值搜索前的初始阈值；同分时优先靠近这个阈值。
NOTEBOOK_INITIAL_THRESHOLD = 0.42
# NOTEBOOK_THRESHOLD_SELECTION_METRIC: 用 CV/OOF 校准后概率挑选最终分类阈值时的主指标。
NOTEBOOK_THRESHOLD_SELECTION_METRIC = "F1"
# NOTEBOOK_THRESHOLD_PROBE_THRESHOLDS: 阈值探测时扫描的阈值列表；会显式包含初始阈值。
NOTEBOOK_THRESHOLD_PROBE_THRESHOLDS = sorted(
    set(np.round(np.linspace(0.30, 0.70, 41), 3).tolist() + [NOTEBOOK_INITIAL_THRESHOLD])
)
# NOTEBOOK_CALIBRATION_METHOD: 概率校准方法；优先用 Isotonic Regression，必要时自动退回 Platt Scaling。
NOTEBOOK_CALIBRATION_METHOD = "isotonic"
NOTEBOOK_MODEL_OUTPUT_PATH = PROJECT_ROOT / "models" / "lightgbm_cuda_model.txt"
NOTEBOOK_PREPROCESSOR_OUTPUT_PATH = PROJECT_ROOT / "models" / "lightgbm_cuda_preprocessor.joblib"
NOTEBOOK_MANIFEST_OUTPUT_PATH = PROJECT_ROOT / "models" / "lightgbm_cuda_inference_assets.json"

NOTEBOOK_LGBM_SEARCH_SPACES = {
    "num_leaves": Integer(48, 127),  # 扩大树结构容量，给复杂边界留出更多叶子空间。
    "learning_rate": Real(8e-3, 3e-2, prior="log-uniform"),  # 放宽 boosting 速度，让搜索同时覆盖更快和更稳的学习率。
    "n_estimators": Integer(2500, 6000),  # 显著拉长 boosting 长度，再交给折内 early stopping 截断。
    "max_depth": Categorical([6, 7, 8, 9]),  # 允许更深的树结构，但仍保留明确上限。
    "subsample": Real(0.80, 1.00),  # 行采样比例略收窄，优先让 GPU 持续吃到更多样本。
    "colsample_bytree": Real(0.80, 1.00),  # 列采样比例同步上调，增加单轮建树的信息密度。
    "min_child_samples": Integer(60, 160),  # 分裂门槛继续抬高，避免大容量下叶子过碎。
    "min_split_gain": Real(0.08, 0.60, prior="log-uniform"),  # 继续拉开分裂增益门槛，抑制边际收益很低的分裂。
    "reg_alpha": Real(5e-2, 1.5, prior="log-uniform"),  # L1 只做温和约束，不再先猛加正则。
    "reg_lambda": Real(2.0, 20.0, prior="log-uniform"),  # L2 同样保留中等强度，重点仍放在结构与 boosting 搜索。
}

NOTEBOOK_CONFIG = cuda_training_support.build_notebook_run_config(
    bayes_n_iter=NOTEBOOK_BAYES_N_ITER,
    cv_folds=NOTEBOOK_CV_FOLDS,
    random_seed=NOTEBOOK_RANDOM_SEED,
    test_size=NOTEBOOK_TEST_SIZE,
    model_n_jobs=NOTEBOOK_MODEL_N_JOBS,
    search_n_jobs=NOTEBOOK_SEARCH_N_JOBS,
    smote_k_neighbors=NOTEBOOK_SMOTE_K_NEIGHBORS,
    smote_sampling_strategy=NOTEBOOK_SMOTE_SAMPLING_STRATEGY,
    early_stopping_rounds=NOTEBOOK_EARLY_STOPPING_ROUNDS,
    early_stopping_validation_fraction=NOTEBOOK_EARLY_STOPPING_VALIDATION_FRACTION,
    calibration_method=NOTEBOOK_CALIBRATION_METHOD,
    initial_threshold=NOTEBOOK_INITIAL_THRESHOLD,
)
NOTEBOOK_CV = StratifiedKFold(
    n_splits=NOTEBOOK_CONFIG.cv_folds,
    shuffle=True,
    random_state=NOTEBOOK_CONFIG.random_seed,
)
DEFAULT_TARGET_COLUMN = cuda_training_support.DEFAULT_TARGET_COLUMN
DATA_PATH = cuda_training_support.resolve_data_path(start_dir=Path.cwd())

print(
    cuda_training_support.format_notebook_run_summary(
        NOTEBOOK_CONFIG,
        data_path=DATA_PATH,
    )
)
print("重采样/权重路径: Fold 内 Borderline-SMOTE 增样 + 重采样后 scale_pos_weight")
print(f"阈值选择指标: {NOTEBOOK_THRESHOLD_SELECTION_METRIC}")
print(f"阈值候选数: {len(NOTEBOOK_THRESHOLD_PROBE_THRESHOLDS)}")
print(f"模型输出路径: {NOTEBOOK_MODEL_OUTPUT_PATH}")
print(f"预处理输出路径: {NOTEBOOK_PREPROCESSOR_OUTPUT_PATH}")
print(f"推理清单路径: {NOTEBOOK_MANIFEST_OUTPUT_PATH}")


ModuleNotFoundError: No module named 'numpy'

## LightGBM（CUDA-only）

- 第一个代码单元负责 import、路径修正、单路径训练配置、阈值候选、概率校准配置，以及模型输出路径。
- 第二个代码单元只做数据加载、`RETENTION_TIME` 清洗诊断、训练/测试集切分，不启动训练。
- 第三个代码单元只跑一轮 `BayesSearchCV`：每个外层训练折都会在折内先切 early stopping 验证集，再只对训练子集做 Borderline-SMOTE 增样，并基于增样后的新比例自动计算 `scale_pos_weight`。
- 第四个代码单元会基于训练集 CV/OOF 原始概率先做严格概率校准，再在校准后的 OOF 概率上完成阈值标定，最后才评估独立测试集并导出推理资产。
- 外层验证集、OOF 与最终测试集始终保持原始真实分布，不会被 SMOTE 污染。
- LightGBM 训练设备固定为 `cuda`，禁止 CPU fallback。
- 为了提升 RTX 4090 的利用率，单模型默认会分配 4~16 个 CPU 线程给 GPU 喂数；搜索层仍保持单 worker，避免多个 CV worker 同时争抢一张卡。
- 缺失值填充、标准化、Borderline-SMOTE 增样和 `scale_pos_weight` 计算都在 estimator 的 `fit` 内部执行；因此每个 CV 训练折都会独立学习预处理和重采样状态。
- `RETENTION_TIME` 会统一解析：普通数值直接保留，多值文本（如 `17.9 and 18.5`）按均值折叠为单值，真正缺失值保留为缺失，后续再由训练折内众数填补。
- Notebook 训练进度条由 `tqdm.auto` 提供。
- 训练完成后会同时保存 `models/lightgbm_cuda_model.txt`、`models/lightgbm_cuda_preprocessor.joblib` 和 `models/lightgbm_cuda_inference_assets.json`；其中会额外写入概率校准器与阈值元数据。
- 官方当前不支持 Windows 上的 CUDA 版 LightGBM。需要把训练移动到 Linux 或 WSL2，并先执行：

```bash
pip install -r requirements.txt
pip uninstall -y lightgbm
pip install lightgbm --no-binary lightgbm --config-settings=cmake.define.USE_CUDA=ON
```



In [2]:
# 加载数据、切分数据并输出训练前诊断。
random_seed = NOTEBOOK_CONFIG.random_seed

data = cuda_training_support.load_training_dataframe(
    data_path=DATA_PATH,
    random_seed=random_seed,
)
retention_time_raw = data["RETENTION_TIME"].copy()

prepared = cuda_training_support.prepare_lightgbm_training_data(
    data,
    target_column=DEFAULT_TARGET_COLUMN,
    random_state=NOTEBOOK_CONFIG.random_seed,
    test_size=NOTEBOOK_CONFIG.test_size,
    smote_k_neighbors=NOTEBOOK_CONFIG.smote_k_neighbors,
)
X_train = prepared["X_train"]
X_test = prepared["X_test"]
y_train = prepared["y_train"]
y_test = prepared["y_test"]
retention_time_diagnostics = prepared["retention_time_diagnostics"]

print("=== 训练前诊断 ===")
print("样本总数:", len(data))
print("标签分布:")
print(data[DEFAULT_TARGET_COLUMN].value_counts())
print()
print("转换前 RETENTION_TIME 的示例值：")
print(retention_time_raw.head())
print()
if retention_time_diagnostics is not None:
    print("RETENTION_TIME 清洗诊断：")
    print(f"严格数值转换后的缺失数: {retention_time_diagnostics['strict_missing_count']}")
    print(f"清洗后的缺失数        : {retention_time_diagnostics['cleaned_missing_count']}")
    print(f"从文本中恢复的记录数  : {retention_time_diagnostics['recovered_from_text_count']}")
    print(f"多值文本记录数        : {retention_time_diagnostics['multi_value_count']}")
    print(f"原始真实缺失数        : {retention_time_diagnostics['original_missing_count']}")
    print(f"仍无法解析的非空记录数: {retention_time_diagnostics['unparsed_non_missing_count']}")
    print("多值文本示例：")
    print(retention_time_diagnostics["multi_value_examples"] or ["<none>"])
    print()
print("训练集形状（原始；fold 内会先切 early stopping，再只对训练子集做 Borderline-SMOTE 增样）:", X_train.shape)
print("测试集形状:", X_test.shape)
print("训练集标签分布（原始）:")
print(pd.Series(y_train).value_counts())
print()
print("测试集标签分布:")
print(pd.Series(y_test).value_counts())
print()

lgbm_version = cuda_training_support.validate_lightgbm_cuda_build(
    random_state=NOTEBOOK_CONFIG.random_seed,
)
print("LightGBM CUDA preflight:", lgbm_version)


=== 训练前诊断 ===
样本总数: 16806
标签分布:
毒性
0    10920
1     5886
Name: count, dtype: int64

转换前 RETENTION_TIME 的示例值：
0    4.07435
1      2.354
2        NaN
3      7.557
4     13.870
Name: RETENTION_TIME, dtype: str

RETENTION_TIME 清洗诊断：
严格数值转换后的缺失数: 3266
清洗后的缺失数        : 3260
从文本中恢复的记录数  : 6
多值文本记录数        : 6
原始真实缺失数        : 3260
仍无法解析的非空记录数: 0
多值文本示例：
['17.9  and 18.5']

训练集形状（原始；fold 内会先切 early stopping，再只对训练子集做 Borderline-SMOTE 增样）: (14285, 14)
测试集形状: (2521, 14)
训练集标签分布（原始）:
毒性
0    9282
1    5003
Name: count, dtype: int64

测试集标签分布:
毒性
0    1638
1     883
Name: count, dtype: int64



[LightGBM] [Fatal] CUDA Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_CUDA=1


RuntimeError: 当前 LightGBM 不能以 CUDA 模式训练。
已检测到 lightgbm==4.6.0，但它不是启用 USE_CUDA=1 的构建。
项目已禁用任何静默 fallback。
检测到当前系统为 Linux。请确认 NVIDIA CUDA 环境已可用，再继续安装 LightGBM。

Python 包源码安装命令:
  pip install lightgbm --no-binary lightgbm --config-settings=cmake.define.USE_CUDA=ON

如需先手工编译 LightGBM，可使用:
git clone --recursive https://github.com/microsoft/LightGBM
cd LightGBM
cmake -B build -S . -DUSE_CUDA=ON
cmake --build build -j4

In [ ]:
# 单路径训练块：Fold 内 Borderline-SMOTE 增样，再按增样后比例自动计算 scale_pos_weight。
metric_order = [
    "AUC",
    "Accuracy",
    "Balanced Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Specificity",
]


def print_metric_block(title, metrics):
    print(title)
    for metric in metric_order:
        print(f"{metric:<18}: {metrics[metric]:.4f}")
    print()


lgb_model = cuda_training_support.build_lgbm_classifier(
    random_state=NOTEBOOK_CONFIG.random_seed,
    model_n_jobs=NOTEBOOK_CONFIG.model_n_jobs,
    smote_k_neighbors=NOTEBOOK_CONFIG.smote_k_neighbors,
    smote_sampling_strategy=NOTEBOOK_CONFIG.smote_sampling_strategy,
    early_stopping_rounds=NOTEBOOK_CONFIG.early_stopping_rounds,
    early_stopping_validation_fraction=NOTEBOOK_CONFIG.early_stopping_validation_fraction,
)

bayes_search = BayesSearchCV(
    estimator=lgb_model,
    search_spaces=NOTEBOOK_LGBM_SEARCH_SPACES,
    n_iter=NOTEBOOK_CONFIG.bayes_n_iter,
    cv=NOTEBOOK_CV,
    scoring=NOTEBOOK_BAYES_SCORING,
    n_jobs=NOTEBOOK_CONFIG.search_n_jobs,
    verbose=NOTEBOOK_BAYES_VERBOSE,
    random_state=NOTEBOOK_CONFIG.random_seed,
)

progress_bar = tqdm(
    total=NOTEBOOK_CONFIG.bayes_n_iter,
    desc=NOTEBOOK_TQDM_DESC,
    unit="iter",
)
progress_state = {"completed": 0}


def update_training_progress(_optim_result):
    progress_state["completed"] += 1
    progress_bar.update(1)
    progress_bar.set_postfix(completed=progress_state["completed"], refresh=False)
    return False


try:
    bayes_search.fit(X_train, y_train, callback=update_training_progress)
finally:
    progress_bar.close()

best_lgb = bayes_search.best_estimator_
best_params = dict(bayes_search.best_params_)
best_cv_auc = float(bayes_search.best_score_)
best_raw_test_proba = best_lgb.predict_proba(X_test)[:, 1]
initial_test_metrics = cuda_training_support.compute_binary_classification_metrics(
    y_true=y_test,
    positive_proba=best_raw_test_proba,
    threshold=NOTEBOOK_CONFIG.initial_threshold,
)

print("最佳参数组合:", best_params)
print("最佳交叉验证AUC:", best_cv_auc)
print("最终 refit 前训练集标签分布:", getattr(best_lgb, "fit_class_counts_", {}))
print("折内训练子集标签分布:", getattr(best_lgb, "train_split_class_counts_", {}))
print("折内 early stopping 验证集标签分布:", getattr(best_lgb, "eval_split_class_counts_", {}))
print("Borderline-SMOTE 元数据:", getattr(best_lgb, "resampling_metadata_", {}))
print("重采样前 scale_pos_weight:", getattr(best_lgb, "pre_resample_scale_pos_weight_", None))
print("重采样后 scale_pos_weight:", getattr(best_lgb, "effective_scale_pos_weight_", None))
print("best_iteration:", getattr(best_lgb, "best_iteration_", None))
print(f"下方测试集指标仅作为初始阈值 {NOTEBOOK_CONFIG.initial_threshold:.2f} 的部署起点，不参与阈值选择。")
print_metric_block(
    f"=== 测试集性能（原始概率，threshold={NOTEBOOK_CONFIG.initial_threshold:.2f}） ===",
    initial_test_metrics,
)

plt.figure(figsize=(10, 6))
lgb.plot_importance(best_lgb.booster_, max_num_features=20)
plt.tight_layout()
plt.show()



In [ ]:
# CV/OOF 概率校准与阈值标定块：只参考训练集 OOF，不看 refit 后训练集分数。
allowed_threshold_metrics = {
    "Accuracy",
    "Balanced Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Specificity",
}
if NOTEBOOK_THRESHOLD_SELECTION_METRIC not in allowed_threshold_metrics:
    raise ValueError(
        f"NOTEBOOK_THRESHOLD_SELECTION_METRIC 必须属于 {sorted(allowed_threshold_metrics)}"
    )

raw_oof_train_proba = np.zeros(len(y_train), dtype=float)
oof_progress = tqdm(total=NOTEBOOK_CONFIG.cv_folds, desc="OOF Raw Probability CV", unit="fold")

try:
    for fold_idx, (train_idx, valid_idx) in enumerate(NOTEBOOK_CV.split(X_train, y_train), start=1):
        fold_model = clone(best_lgb)
        X_fold_train = X_train.iloc[train_idx].reset_index(drop=True)
        y_fold_train = y_train.iloc[train_idx].reset_index(drop=True)
        X_fold_valid = X_train.iloc[valid_idx].reset_index(drop=True)

        fold_model.fit(X_fold_train, y_fold_train)
        raw_oof_train_proba[valid_idx] = fold_model.predict_proba(X_fold_valid)[:, 1]
        oof_progress.update(1)
        oof_progress.set_postfix(fold=fold_idx, refresh=False)
finally:
    oof_progress.close()

probability_calibration_bundle = cuda_training_support.fit_probability_calibrator(
    y_true=y_train,
    positive_proba=raw_oof_train_proba,
    method=NOTEBOOK_CONFIG.calibration_method,
    random_state=NOTEBOOK_CONFIG.random_seed,
)
calibrated_oof_train_proba = probability_calibration_bundle["calibrated_positive_proba"]
best_calibrated_test_proba = cuda_training_support.apply_probability_calibrator(
    best_raw_test_proba,
    probability_calibration_bundle,
)

threshold_probe = cuda_training_support.probe_binary_classification_thresholds(
    y_true=y_train,
    positive_proba=calibrated_oof_train_proba,
    thresholds=NOTEBOOK_THRESHOLD_PROBE_THRESHOLDS,
)

top_f1_thresholds = threshold_probe.sort_values(
    by=["F1", "Balanced Accuracy", "Recall", "threshold"],
    ascending=[False, False, False, True],
).head(10)
top_balanced_thresholds = threshold_probe.sort_values(
    by=["Balanced Accuracy", "F1", "Recall", "threshold"],
    ascending=[False, False, False, True],
).head(10)
selected_threshold_row = cuda_training_support.select_binary_classification_threshold(
    threshold_probe,
    primary_metric=NOTEBOOK_THRESHOLD_SELECTION_METRIC,
    initial_threshold=NOTEBOOK_CONFIG.initial_threshold,
)
SELECTED_THRESHOLD = float(selected_threshold_row["threshold"])

initial_oof_metrics = cuda_training_support.compute_binary_classification_metrics(
    y_true=y_train,
    positive_proba=calibrated_oof_train_proba,
    threshold=NOTEBOOK_CONFIG.initial_threshold,
)
selected_oof_metrics = cuda_training_support.compute_binary_classification_metrics(
    y_true=y_train,
    positive_proba=calibrated_oof_train_proba,
    threshold=SELECTED_THRESHOLD,
)
selected_test_metrics = cuda_training_support.compute_binary_classification_metrics(
    y_true=y_test,
    positive_proba=best_calibrated_test_proba,
    threshold=SELECTED_THRESHOLD,
)
selected_test_confusion_matrix = selected_test_metrics["confusion_matrix"]
threshold_selection_metadata = {
    "selection_source": "cv_oof_calibrated",
    "selection_metric": NOTEBOOK_THRESHOLD_SELECTION_METRIC,
    "initial_threshold": NOTEBOOK_CONFIG.initial_threshold,
    "selected_threshold_row": selected_threshold_row,
    "candidate_threshold_count": len(NOTEBOOK_THRESHOLD_PROBE_THRESHOLDS),
    "oof_metrics_at_selected_threshold": {
        key: value
        for key, value in selected_oof_metrics.items()
        if key != "confusion_matrix"
    },
}

saved_artifacts = cuda_training_support.save_lightgbm_inference_artifacts(
    estimator=best_lgb,
    prepared=prepared,
    model_path=NOTEBOOK_MODEL_OUTPUT_PATH,
    preprocessor_path=NOTEBOOK_PREPROCESSOR_OUTPUT_PATH,
    manifest_path=NOTEBOOK_MANIFEST_OUTPUT_PATH,
    target_column=DEFAULT_TARGET_COLUMN,
    data_path=DATA_PATH,
    random_seed=NOTEBOOK_CONFIG.random_seed,
    test_size=NOTEBOOK_CONFIG.test_size,
    smote_k_neighbors=NOTEBOOK_CONFIG.smote_k_neighbors,
    smote_sampling_strategy=NOTEBOOK_CONFIG.smote_sampling_strategy,
    scoring=NOTEBOOK_BAYES_SCORING,
    classification_threshold=SELECTED_THRESHOLD,
    probability_calibration_bundle=probability_calibration_bundle,
    threshold_selection_metadata=threshold_selection_metadata,
)

print("=== 概率校准摘要 ===")
print("请求的校准方法:", probability_calibration_bundle["requested_method"])
print("实际使用的校准方法:", probability_calibration_bundle["method"])
if probability_calibration_bundle.get("fallback_reason"):
    print("校准回退原因:", probability_calibration_bundle["fallback_reason"])
print("OOF 原始概率指标:", probability_calibration_bundle["raw_metrics"])
print("OOF 校准后概率指标:", probability_calibration_bundle["calibrated_metrics"])
print()
print("=== 训练集 OOF 阈值探测：按 F1 排名前 10 ===")
display(top_f1_thresholds)
print("=== 训练集 OOF 阈值探测：按 Balanced Accuracy 排名前 10 ===")
display(top_balanced_thresholds)
print(f"选定阈值: {SELECTED_THRESHOLD:.3f}")
print(f"阈值选择指标: {NOTEBOOK_THRESHOLD_SELECTION_METRIC}")
print_metric_block(
    f"=== 训练集 OOF 性能（校准后概率，threshold={NOTEBOOK_CONFIG.initial_threshold:.2f}） ===",
    initial_oof_metrics,
)
print_metric_block("=== 训练集 OOF 性能（校准后概率，选定阈值） ===", selected_oof_metrics)
print_metric_block("=== 测试集性能（校准后概率，选定阈值） ===", selected_test_metrics)
print("测试集混淆矩阵（校准后概率，选定阈值）:")
print(selected_test_confusion_matrix)
print(f"模型已保存到: {saved_artifacts['model_path']}")
print(f"预处理包已保存到: {saved_artifacts['preprocessor_path']}")
print(f"推理清单已保存到: {saved_artifacts['manifest_path']}")

